# 01 — Exploratory Data Analysis (EDA)

**Tujuan:** memahami data sebelum memilih model: jumlah label, ketidakseimbangan
kelas, panjang teks, duplikat, dan dampak normalisasi. EDA yang jujur mencegah
kita menyimpulkan "model bagus" padahal datanya bermasalah.

**Prasyarat:** `data/raw/re_dataset.csv` sudah ada (lihat `docs/dataset.md`).
Jika belum ada, notebook ini menampilkan pesan dan melewati analisis, bukan
error.

**Estimasi waktu:** 5 menit.

**Lisensi data:** CC BY-NC-SA 4.0. Sitasi wajib: Ibrohim & Budi (2019),
*Multi-label Hate Speech and Abusive Language Detection in Indonesian Twitter*,
ALW3. Notebook ini **tidak** menampilkan isi tweet mentah ke output, agar tidak
membocorkan teks pengguna ke file yang di-commit.

## Glosarium

- **EDA**: pemeriksaan statistik dan pola data sebelum modeling.
- **Kelas positif**: di proyek ini `bully = 1` (HS atau Abusive).
- **Imbalanced**: jumlah kelas tidak seimbang; accuracy bisa menyesatkan.
- **Duplikat**: teks sama setelah normalisasi; jika dibiarkan bisa bocor ke
  train dan test.
- **Token**: potongan kata yang dihasilkan tokenizer BERT.
- **Data leakage**: informasi test ikut terpakai saat training, membuat skor
  terlihat bagus tetapi palsu.

## 1. Muat data

Kita memakai `read_csv_auto` dari `src/`, bukan `pandas.read_csv` langsung.
Alasannya: file asli bukan UTF-8 murni. Fungsi ini mencoba UTF-8 strict lalu
mengganti byte rusak dengan peringatan, sehingga notebook tidak crash dan kita
tahu berapa karakter yang hilang.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from cyberbully.config import DATA_RAW, LABEL_COLUMN, TEXT_COLUMN
from cyberbully.data import dedup, load_csv, map_labels
from cyberbully.preprocess import normalize_text

DATA_PATH = DATA_RAW / "re_dataset.csv"
df = None

if not DATA_PATH.exists():
    print("Dataset belum ada:", DATA_PATH)
    print("Ikuti docs/dataset.md, lalu restart & run all.")
else:
    df = load_csv(DATA_PATH, "Tweet", ["HS", "Abusive"], keep_cols=["HS", "Abusive"])
    print("shape:", df.shape)
    print("kolom:", list(df.columns))

## 2. Distribusi label

`HS` berarti ujaran kebencian, `Abusive` berarti bahasa kasar/hinaan. Proyek ini
memakai definisi **union**: sebuah tweet dianggap bully bila salah satunya 1.
Grafik membantu melihat apakah kelas minoritas terlalu kecil, yang akan membuat
recall kelas bully rapuh.

In [ ]:
if df is not None:
    counts = {
        "HS": int(df["HS"].sum()),
        "Abusive": int(df["Abusive"].sum()),
        "Union (HS atau Abusive)": int(((df["HS"] == 1) | (df["Abusive"] == 1)).sum()),
    }
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(list(counts), list(counts.values()), color=["#8c564b", "#1f77b4", "#2ca02c"])
    ax.set_ylabel("jumlah tweet")
    ax.set_title("Jumlah tweet positif per definisi label")
    for index, value in enumerate(counts.values()):
        ax.text(index, value + 100, f"{value:,}", ha="center")
    plt.show()
    print(counts)

## 3. Panjang teks

Panjang teks menentukan `max_length` tokenizer. Teks yang sangat panjang akan
terpotong; teks pendek bisa kehilangan konteks. Kita melihat distribusi jumlah
kata, bukan jumlah karakter, karena tokenizer memotong berdasarkan token.

In [ ]:
if df is not None:
    word_counts = df[TEXT_COLUMN].astype(str).str.split().str.len()
    print(word_counts.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1))

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.hist(word_counts, bins=40)
    ax.set_xlabel("jumlah kata per tweet")
    ax.set_ylabel("frekuensi")
    ax.set_title("Distribusi panjang teks")
    plt.show()

## 4. Duplikat dan normalisasi

Dua tweet bisa berbeda hanya karena spasi, huruf besar, atau URL. Normalisasi
ringan menyamakan bentuk tersebut tanpa menghapus emoji/tanda baca. Setelah itu
kita hitung duplikat; duplikat harus dibuang **sebelum** split agar tidak ada
teks yang muncul di train sekaligus test.

In [ ]:
if df is not None:
    texts = df[TEXT_COLUMN].astype(str)
    normalized = texts.map(normalize_text)
    print("duplikat persis      :", int(texts.duplicated().sum()))
    print("duplikat setelah norm:", int(normalized.duplicated().sum()))

## 5. Kelas setelah dedup

Sekarang kita terapkan keputusan label union dan deduplikasi, lalu lihat
keseimbangan kelas final. Angka inilah yang menjadi dasar pemilihan
`--class-weights` saat training. Accuracy saja tidak cukup: model yang selalu
menebak kelas mayoritas bisa terlihat tinggi padahal F1 bully nol.

In [ ]:
if df is not None:
    mapped = map_labels(df, ["1"], label_cols=["HS", "Abusive"], mode="any")
    clean = dedup(mapped)
    counts = clean[LABEL_COLUMN].value_counts().sort_index()
    ratio = clean[LABEL_COLUMN].mean()
    print("baris sebelum dedup:", len(mapped))
    print("baris setelah dedup:", len(clean))
    print("positif:", int(counts.get(1, 0)), "negatif:", int(counts.get(0, 0)))
    print(f"rasio bully: {ratio:.3f}")

## 6. Pratinjau split (tanpa menyimpan)

Fungsi `split_dataset` memakai stratifikasi dan seed 42, jadi proporsi kelas
terjaga di tiap split. Di sini kita hanya menghitung ukurannya; penulisan CSV
dilakukan oleh CLI data agar EDA tidak mengubah artefak.

In [ ]:
if df is not None:
    from cyberbully.data import split_dataset

    splits = split_dataset(clean)
    for name, part in splits.items():
        print(f"{name:5s} {len(part):5d} baris | positif={int(part[LABEL_COLUMN].sum())}")

## Ringkasan

Yang perlu dibawa ke laporan:

1. Definisi label union menghasilkan sekitar 55% kelas positif; cukup seimbang,
   tetapi `--class-weights` tetap disediakan untuk eksperimen.
2. Panjang teks mayoritas pendek; `max_length=128` aman dan `64` bisa dipakai
   untuk smoke test.
3. Ada ratusan duplikat setelah normalisasi; dedup sebelum split wajib.
4. Encoding file asli rusak sebagian dan emoji ter-escape; pipeline sudah
   menanganinya, dan jumlah karakter rusak dilaporkan sebagai peringatan.
5. EDA tidak membuktikan performa model. Klaim performa hanya boleh muncul dari
   evaluasi validation/test yang terdokumentasi.